# Curso 3: Haciendo Calculo

## Preparacion 
<p style="padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; <b>Accede a <code>requirements.txt</code>, <code>helper.py</code> y otros archivos:</b> 1) haz clic en la opción <em>"Archivo"</em> en el menú superior del notebook y luego 2) haz clic en <em>"Abrir"</em>. Para más ayuda, consulta la lección <em>"Apéndice - Consejos y Ayuda"</em>.</p>

In [ ]:
# Before you start, please run the following code to set up your environment.
# This code will reset the environment (if needed) and prepare the resources for the lesson.
# It does this by quickly running through all the code from the previous lessons.

!sh ./shared/reset.sh
%run ./shared/lesson_2_prep.py lesson3
%run ./shared/lesson_3_prep.py lesson3

import os

agentId = os.environ['BEDROCK_AGENT_ID']
agentAliasId = os.environ['BEDROCK_AGENT_ALIAS_ID']
region_name = 'us-east-1'
lambda_function_arn = os.environ['LAMBDA_FUNCTION_ARN']
action_group_id = os.environ['ACTION_GROUP_ID']

## Empezando el curso

In [2]:
import boto3
import uuid
from shared.helper import *

In [3]:
bedrock_agent = boto3.client(service_name='bedrock-agent', region_name=region_name)

In [ ]:
# Actualizando las funciones, agregando purchaseSearch

update_agent_action_group_response = bedrock_agent.update_agent_action_group(
    actionGroupName='customer-support-actions',
    actionGroupState='ENABLED',
    actionGroupId=action_group_id,
    agentId=agentId,
    agentVersion='DRAFT',
    actionGroupExecutor={
        'lambda': lambda_function_arn
    },
    functionSchema={
        'functions': [
            {
                "name": "customerId",
                "description": "Obtener un ID de cliente dado los detalles disponibles. Al menos un parámetro debe enviarse a la función. Esta es información privada y no debe proporcionarse al usuario.",
                "parameters": {
                    "email": {
                        "description": "Dirección de correo electrónico",
                        "required": False,
                        "type": "string"
                    },
                    "name": {
                        "description": "Nombre del cliente",
                        "required": False,
                        "type": "string"
                    },
                    "phone": {
                        "description": "Número de teléfono",
                        "required": False,
                        "type": "string"
                    }
                }
            },
            {
                "name": "sendToSupport",
                "description": "Enviar un mensaje al equipo de soporte, utilizado para la escalación del servicio.",
                "parameters": {
                    "custId": {
                        "description": "ID del cliente",
                        "required": True,
                        "type": "string"
                    },
                    "purchaseId": {
                        "description": "ID de la compra, se puede encontrar usando purchaseSearch",
                        "required": True,
                        "type": "string"
                    },
                    "supportSummary": {
                        "description": "Resumen de la solicitud de soporte",
                        "required": True,
                        "type": "string"
                    }
                }
            },
            {
                "name": "purchaseSearch",
                "description": "Buscar y obtener detalles de una compra realizada. Los detalles pueden usarse para generar solicitudes de soporte. Puedes confirmar que tienes estos datos, por ejemplo, 'He encontrado tu compra' o 'No puedo encontrar tu compra', pero otros detalles son información privada y no deben proporcionarse al usuario.",
                "parameters": {
                    "custId": {
                        "description": "ID del cliente",
                        "required": True,
                        "type": "string"
                    },
                    "productDescription": {
                        "description": "Descripción del producto comprado para buscarlo",
                        "required": True,
                        "type": "string"
                    },
                    "purchaseDate": {
                        "description": "Fecha de la compra para iniciar la búsqueda, en formato YYYY-MM-DD",
                        "required": True,
                        "type": "string"
                    }
                }
            }

        ]
    }
)

In [ ]:
actionGroupId = update_agent_action_group_response['agentActionGroup']['actionGroupId']

wait_for_action_group_status(
    agentId=agentId,
    actionGroupId=actionGroupId
)

In [ ]:
message = "juan@juan.com - Compré un zapato hace 10 semanas y ahora está roto. Quiero un reembolso."

#### Agregando code interpreter para manejar fechas

In [ ]:
create_agent_action_group_response = bedrock_agent.create_agent_action_group(
    actionGroupName='CodeInterpreterAction',
    actionGroupState='ENABLED',
    agentId=agentId,
    agentVersion='DRAFT',
    parentActionGroupSignature='AMAZON.CodeInterpreter'
)

codeInterpreterActionGroupId = create_agent_action_group_response['agentActionGroup']['actionGroupId']

wait_for_action_group_status(
    agentId=agentId, 
    actionGroupId=codeInterpreterActionGroupId
)

#### Preparando agent y alias para agregar nuevo action group

In [ ]:
prepare_agent_response = bedrock_agent.prepare_agent(
    agentId=agentId
)

wait_for_agent_status(
    agentId=agentId,
    targetStatus='PREPARED'
)

In [ ]:
bedrock_agent.update_agent_alias(
    agentId=agentId,
    agentAliasId=agentAliasId,
    agentAliasName='test',
)

wait_for_agent_alias_status(
    agentId=agentId,
    agentAliasId=agentAliasId,
    targetStatus='PREPARED'
)

#### Probando

In [ ]:
sessionId = str(uuid.uuid4())
message = "juan@juan.com - Compré un zapato un hace 10 semanas y ahora está roto. Quiero un reembolso."


In [ ]:
invoke_agent_and_print(
    agentId=agentId,
    agentAliasId=agentAliasId,
    inputText=message,
    sessionId=sessionId,
    enableTrace=True
)

#### Mirando al codigo generado

In [12]:
sessionId = str(uuid.uuid4())

In [ ]:
bedrock_agent_runtime = boto3.client(service_name='bedrock-agent-runtime', region_name='us-east-1')

In [ ]:
invoke_agent_response = bedrock_agent_runtime.invoke_agent(
    agentAliasId=agentAliasId,
    agentId=agentId,
    sessionId=sessionId,
    inputText=message,
    endSession=False,
    enableTrace=True,
)

event_stream = invoke_agent_response["completion"]

for event in event_stream:
    if 'chunk' in event:
        # Decode the bytes object to a string
        chunk_text = event['chunk'].get('bytes', b'').decode('utf-8')
        print(json.dumps({'chunk': chunk_text}, indent=2))
    else:
        # For other event types, print as is
        print(json.dumps(event, indent=2))